# Quantum teleportation with Qiskit

This notebook implements quantum teleportation and verifies it for four input states.

In [ ]:
# Run this cell in Google Colab.
%pip install -q qiskit qiskit-aer

In [2]:
from qiskit import ClassicalRegister, QuantumCircuit, QuantumRegister, transpile
from qiskit_aer import AerSimulator

In [3]:
def teleportation_circuit(preparation):
    if preparation.num_qubits != 1 or preparation.num_clbits:
        raise ValueError("preparation must be a one-qubit circuit without classical bits")

    qubits = QuantumRegister(3, "q")
    source_result = ClassicalRegister(1, "source")
    pair_result = ClassicalRegister(1, "pair")
    verification = ClassicalRegister(1, "verify")
    circuit = QuantumCircuit(qubits, source_result, pair_result, verification)

    # Prepare the state to be teleported and the shared Bell pair.
    circuit.compose(preparation, [qubits[0]], inplace=True)
    circuit.h(qubits[1])
    circuit.cx(qubits[1], qubits[2])

    # Perform a Bell-basis measurement on the sender's qubits.
    circuit.cx(qubits[0], qubits[1])
    circuit.h(qubits[0])
    circuit.measure(qubits[0], source_result[0])
    circuit.measure(qubits[1], pair_result[0])

    # Apply the receiver's corrections selected by the two classical bits.
    with circuit.if_test((pair_result, 1)):
        circuit.x(qubits[2])
    with circuit.if_test((source_result, 1)):
        circuit.z(qubits[2])

    # Applying U† to the received U|0> must return the qubit to |0>.
    circuit.compose(preparation.inverse(), [qubits[2]], inplace=True)
    circuit.measure(qubits[2], verification[0])
    return circuit

def run_teleportation(preparation, shots=100, seed=None):
    simulator = AerSimulator()
    compiled = transpile(teleportation_circuit(preparation), simulator)
    result = simulator.run(compiled, shots=shots, seed_simulator=seed).result()
    return dict(result.get_counts())

def verification_success(counts):
    total = sum(counts.values())
    successful = sum(count for outcome, count in counts.items() if outcome.split()[0] == "0")
    return successful / total

In [4]:
preparations = {}

preparations["|0>"] = QuantumCircuit(1)

preparations["|1>"] = QuantumCircuit(1)
preparations["|1>"].x(0)

preparations["|+>"] = QuantumCircuit(1)
preparations["|+>"].h(0)

preparations["|+i>"] = QuantumCircuit(1)
preparations["|+i>"].h(0)
preparations["|+i>"].s(0)

In [5]:
teleportation_circuit(preparations["|+i>"]).draw()

┌───┐┌───┐     ┌───┐┌─┐                                              »
     q_0: ┤ H ├┤ S ├──■──┤ H ├┤M├──────────────────────────────────────────────»
          ├───┤└───┘┌─┴─┐└┬─┬┘└╥┘                                              »
     q_1: ┤ H ├──■──┤ X ├─┤M├──╫───────────────────────────────────────────────»
          └───┘┌─┴─┐└───┘ └╥┘  ║ ┌────── ┌───┐ ───────┐ ┌────── ┌───┐ ───────┐ »
     q_2: ─────┤ X ├───────╫───╫─┤ If-0  ┤ X ├  End-0 ├─┤ If-0  ┤ Z ├  End-0 ├─»
               └───┘       ║   ║ └──╥─── └───┘ ───────┘ └──╥─── └───┘ ───────┘ »
                           ║   ║    ║                   ┌──╨──┐                »
source: 1/═════════════════╬═══╩════╬═══════════════════╡ 0x1 ╞════════════════»
                           ║   0 ┌──╨──┐                └─────┘                »
  pair: 1/═════════════════╩═════╡ 0x1 ╞═══════════════════════════════════════»
                           0     └─────┘                                       »
verify: 1/═════════════════════════════════════════════════════════════════════»
                                                                               »
«                         
«     q_0: ───────────────
«                         
«     q_1: ───────────────
«          ┌─────┐┌───┐┌─┐
«     q_2: ┤ Sdg ├┤ H ├┤M├
«          └─────┘└───┘└╥┘
«source: 1/═════════════╬═
«                       ║ 
«  pair: 1/═════════════╬═
«                       ║ 
«verify: 1/═════════════╩═
«                       0

In [6]:
for name, preparation in preparations.items():
    counts = run_teleportation(preparation, shots=100, seed=15015)
    print(name, counts, "success =", verification_success(counts))

|0> {'0 1 1': 26, '0 0 0': 24, '0 1 0': 28, '0 0 1': 22} success = 1.0
|1> {'0 1 1': 26, '0 0 0': 24, '0 1 0': 28, '0 0 1': 22} success = 1.0
|+> {'0 1 1': 26, '0 0 0': 24, '0 1 0': 28, '0 0 1': 22} success = 1.0
|+i> {'0 1 1': 26, '0 0 0': 24, '0 1 0': 28, '0 0 1': 22} success = 1.0
